In [2]:
import numpy as np

# Jeu de données
# Chaque élément contient :
# (phrase française, traduction anglaise)
data = [
    ("bonjour", "hello"),
    ("merci", "thanks"),
    ("oui", "yes"),
    ("non", "no"),
    ("bonne nuit", "good night"),
    ("je mange", "i eat"),
    ("je cours", "i run"),
    ("je dors", "i sleep"),
    ("chat noir", "black cat"),
    ("chien blanc", "white dog")
]

# récupérer les mots français sans doublons
fr_words = set()

for fr, en in data:
    for word in fr.split():
        fr_words.add(word)

# trier les mots
fr_words = sorted(list(fr_words))

# chaque traduction complète est une classe
en_words = sorted(list(set(en for fr, en in data)))

# donner un numéro à chaque mot
fr_word_to_index = {word: i for i, word in enumerate(fr_words)}

print("Nombre de mots français :", len(fr_words))
print("Nombre de traductions :", len(en_words))


# taille du vecteur d'un mot
embedding_size = 8

np.random.seed(42)

# chaque mot reçoit un vecteur
embedding = np.random.randn(len(fr_words), embedding_size) * 0.1


# matrices pour calculer Q, K et V
Wq = np.random.randn(embedding_size, embedding_size) * 0.1
Wk = np.random.randn(embedding_size, embedding_size) * 0.1
Wv = np.random.randn(embedding_size, embedding_size) * 0.1


# couche finale pour faire la prédiction
Wy = np.random.randn(embedding_size, len(en_words)) * 0.1
by = np.zeros((1, len(en_words)))


# softmax transforme les scores en probabilités
def softmax(x):

    x = x - np.max(x, axis=-1, keepdims=True)

    exp_x = np.exp(x)

    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


# calcul du self-attention
def self_attention(sentence):

    # séparer la phrase en mots
    words = sentence.split()

    # récupérer le vecteur de chaque mot
    X = np.array([
        embedding[fr_word_to_index[word]]
        for word in words
    ])

    # calculer Query, Key et Value
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv

    # comparer chaque Query avec toutes les Keys
    scores = Q @ K.T

    # normalisation avec la dimension des vecteurs
    scores = scores / np.sqrt(embedding_size)

    # transformer les scores en poids d'attention
    attention = softmax(scores)

    # utiliser les poids pour récupérer les informations de V
    context = attention @ V

    return context, attention


# forward
def forward(sentence):

    # calculer le self-attention
    context, attention = self_attention(sentence)

    # prendre un seul vecteur pour représenter la phrase
    sentence_vector = np.mean(context, axis=0, keepdims=True)

    # calcul des scores de sortie
    scores = sentence_vector @ Wy + by

    # transformer les scores en probabilités
    y = softmax(scores)

    return y, attention


# calcul de l'erreur
def cross_entropy(y, target):

    return -np.log(y[0, target] + 1e-8)


# paramètres d'entraînement
learning_rate = 0.5
epochs = 1000

for epoch in range(epochs):

    total_loss = 0

    for fr, en in data:

        # faire une prédiction
        y, attention = forward(fr)

        # trouver l'indice de la bonne traduction
        target = en_words.index(en)

        # calculer l'erreur
        loss = cross_entropy(y, target)

        total_loss += loss

        # gradient de la sortie
        dy = y.copy()

        # enlever 1 à la bonne réponse
        dy[0, target] -= 1

        # récupérer le vecteur de la phrase
        context, attention = self_attention(fr)

        # moyenne des vecteurs
        sentence_vector = np.mean(context, axis=0, keepdims=True)

        # calcul des gradients
        dWy = sentence_vector.T @ dy
        dby = dy

        # modifier les poids
        Wy -= learning_rate * dWy
        by -= learning_rate * dby

    # afficher la perte tous les 100 epochs
    if epoch % 100 == 0:
        print("Epoch :", epoch, "Loss :", round(total_loss, 4))


# fonction pour prédire une traduction
def predict(sentence):

    # faire passer la phrase dans le modèle
    y, attention = forward(sentence)

    # prendre la classe avec la plus grande probabilité
    index = np.argmax(y)

    return en_words[index]


# afficher les résultats
print("\nRésultats du Self-Attention :\n")
for fr, en in data:
    # faire la prédiction
    prediction = predict(fr)

    print(fr, "→", prediction, "(attendu :", en + ")")


# afficher les poids d'attention
print("\n\nExemple de Self-Attention :")

sentence = "chat noir"

# calculer l'attention pour cette phrase
context, attention = self_attention(sentence)

# récupérer les mots
words = sentence.split()

print("\nPhrase :", sentence)

print("\nPoids d'attention :")
print("       ", words)

# afficher l'attention de chaque mot
for i in range(len(words)):

    print(words[i], attention[i])


# afficher la traduction finale
print("\nPrédiction :", predict(sentence))

Nombre de mots français : 14
Nombre de traductions : 10
Epoch : 0 Loss : 25.148
Epoch : 100 Loss : 23.6092
Epoch : 200 Loss : 22.1503
Epoch : 300 Loss : 20.8607
Epoch : 400 Loss : 19.7206
Epoch : 500 Loss : 18.7086
Epoch : 600 Loss : 17.8055
Epoch : 700 Loss : 16.9945
Epoch : 800 Loss : 16.2621
Epoch : 900 Loss : 15.597

Résultats du Self-Attention :

bonjour → hello (attendu : hello)
merci → thanks (attendu : thanks)
oui → yes (attendu : yes)
non → no (attendu : no)
bonne nuit → good night (attendu : good night)
je mange → i eat (attendu : i eat)
je cours → i sleep (attendu : i run)
je dors → i sleep (attendu : i sleep)
chat noir → black cat (attendu : black cat)
chien blanc → white dog (attendu : white dog)


Exemple de Self-Attention :

Phrase : chat noir

Poids d'attention :
        ['chat', 'noir']
chat [0.49988064 0.50011936]
noir [0.50007355 0.49992645]

Prédiction : black cat
